# Multi-target benchmark

This notebook runs the controlled downsampling benchmark for multiple target cell types
simultaneously, then aggregates cross-target method rankings.

Unlike `rare_cell_downsampling_benchmark.ipynb` (which evaluates a single target using
pre-computed embeddings), this benchmark **recomputes representations** from scratch after
each downsampling — the scientifically stricter design that tests whether each representation
is still usable after cells have been removed.

Mirrors: `scripts/run_multi_target_benchmark.py`

In [ ]:
import subprocess
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
for _p in [PROJECT_ROOT, PROJECT_ROOT / "src"]:
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))
assert (PROJECT_ROOT / "src" / "rarecell").exists(), PROJECT_ROOT
PROJECT_ROOT

## Required inputs

| Input | Description |
|---|---|
| `config/multi_target_config.yaml` | Config: `dataset_path`, `label_column`, `target_cell_types` (null = auto), `fractions`, `seeds`, `representations`, `k_neighbors`, `rebuild_representations_after_downsampling` |
| `data/processed/pbmc5k_10x_citeseq_representations.h5ad` | AnnData with raw counts (representations are rebuilt from scratch by this script, so pre-computed obsm keys are not required but must be present for the run to validate inputs) |

Note: `rebuild_representations_after_downsampling` must be `true` in the multi-target config.
The script raises an error if it is set to `false`.

Use `--max-cells 3000 --fractions 0.1 --seeds 0` for a fast debug run.

## Canonical script command

```bash
python scripts/run_multi_target_benchmark.py --config config/multi_target_config.yaml
```

Debug (fast) command:
```bash
python scripts/run_multi_target_benchmark.py --config config/multi_target_config.yaml \
    --fractions 0.1 --seeds 0 --representations rna_pca --max-cells 3000 --verbose
```

Key functions used internally:
- `rarecell.multi_target.select_target_cell_types(labels, requested)` — selects target populations
- `rarecell.benchmark.run_downsampling_benchmark(adata, ...)` — shared benchmark loop per target
- `rarecell.multi_target.summarize_multi_target_metrics(raw)` — seed-aggregated long-format summary
- `rarecell.multi_target.rank_cross_target_methods(summary)` — mean rank, best/worst count per representation
- `rarecell.plotting.plot_multi_target_metric_curve(raw, metric, output_path)` — metric vs. fraction curves
- `rarecell.plotting.plot_cross_target_method_ranking(ranking, output_path)` — ranking bar chart

In [ ]:
result = subprocess.run(
    [
        sys.executable,
        "scripts/run_multi_target_benchmark.py",
        "--config", "config/multi_target_config.yaml",
    ],
    cwd=PROJECT_ROOT,
    text=True,
    capture_output=True,
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise SystemExit(result.returncode)

## Expected outputs

| File | Description |
|---|---|
| `results/metrics/multi_target_benchmark_raw.csv` | One row per (target, representation, fraction, seed) with precision, recall, F1, neighborhood_purity, silhouette |
| `results/tables/multi_target_benchmark_summary.csv` | Seed-aggregated: mean, std, n_seeds per (target, representation, fraction, metric) |
| `results/tables/cross_target_method_ranking.csv` | Mean rank, best-count, worst-count per (representation, metric) across all targets |
| `results/figures/multi_target_f1_curve.png` | F1 vs. retained fraction, one line per representation, faceted by target |
| `results/figures/multi_target_neighborhood_purity_curve.png` | Neighborhood purity vs. retained fraction, same layout |
| `results/figures/cross_target_method_ranking.png` | Bar chart of mean cross-target rank per representation |
| `results/logs/multi_target_benchmark.log` | Run log with timing and condition counts |

In [ ]:
outputs = [
    "results/metrics/multi_target_benchmark_raw.csv",
    "results/tables/multi_target_benchmark_summary.csv",
    "results/tables/cross_target_method_ranking.csv",
    "results/figures/multi_target_f1_curve.png",
    "results/figures/multi_target_neighborhood_purity_curve.png",
    "results/figures/cross_target_method_ranking.png",
]
[(path, (PROJECT_ROOT / path).exists()) for path in outputs]

## Summary

In [ ]:
print("Generated outputs:")
for path in outputs:
    p = PROJECT_ROOT / path
    status = "OK" if p.exists() else "MISSING"
    print(f"  [{status}] {path}")
print()
print("Deviations from run_multi_target_benchmark.py:")
print("  - No file-level logging (script logs to results/logs/multi_target_benchmark.log).")
print("  - The script caches outputs and reuses them when the grid is unchanged and")
print("    output files are fresh. Re-run the script with --force to recompute.")
print()
print("Note: this benchmark rebuilds representations after each downsampling (slow).")
print("Single-target fast evaluation is in rare_cell_downsampling_benchmark.ipynb.")